# 04. IRAC-V GraphRAG Framework — 4-Stage Ablation Study
**수정 사항 (v3 — KG 효용 공정 실험)**:
1. ★ Flat RAG chunk 수 증대: 법령 원문만 포함, 최소길이 20자
2. ★ V-Agent 정답지(GT) 제거: 내부 일관성 검증만 수행
3. ★ Penalty Accuracy 매칭 로직 정밀화
4. ★ None severity 시나리오 평가 보정
5. ★★ **S2/S3 공정 실험 설계**: S1/S2 프롬프트에서 심각도 기준표 제거 → KG 구조 효용 측정
6. ★★ **시나리오 100건 확대**: DTG 실데이터 기반 시나리오 자동 생성
7. ★★ **Multi-hop Recall 메트릭 추가**: KG 관계 추론 효용 직접 측정

In [1]:
import os, sys, json, time, datetime, re
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from config import (load_json, save_json, LLM_MODEL, get_severity_level,
                    SEVERITY_TABLE,
                    JSON_DIR, PENALTY_DIR, SCENARIO_DIR, KG_DIR, RESULTS_DIR, FIGURES_DIR)

# ★ VIOLATION_MAPPING은 JSON에서 로드 (config.py 10개 → JSON 13개)
VIOLATION_MAPPING = load_json(JSON_DIR / 'violation_article_mapping.json')

# ── Neo4j ──
from neo4j import GraphDatabase
NEO4J_URI  = os.getenv('NEO4J_URI', '')
NEO4J_USER = os.getenv('NEO4J_USER', '')
NEO4J_PW   = os.getenv('NEO4J_PW', '')
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PW))

def run_q(query, params=None):
    with driver.session() as s:
        return [dict(r) for r in s.run(query, params or {})]

# ── LLM ──
import anthropic
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

def call_llm(prompt, system='', max_tokens=2000, temperature=0.0):
    if not system:
        system = '당신은 한국 도로교통법 전문가입니다. 반드시 순수 JSON만 응답하세요.'
    for attempt in range(3):
        try:
            msg = client.messages.create(
                model=LLM_MODEL, max_tokens=max_tokens, temperature=temperature,
                system=system, messages=[{'role':'user','content':prompt}])
            return msg.content[0].text.replace('```json','').replace('```','').strip()
        except Exception as e:
            if 'rate' in str(e).lower(): time.sleep(5*(attempt+1))
            else: raise
    raise Exception('LLM 3회 실패')

def parse_json(text):
    """★ v5: 강건한 JSON 파싱 — unknown 발생 최소화."""
    if not text or not text.strip():
        return None
    # 1차: 표준 파싱
    try:
        start = text.find('{'); end = text.rfind('}')+1
        if start >= 0 and end > start:
            return json.loads(text[start:end])
    except: pass
    # 2차: 줄바꿈·특수문자 정리 후 재시도
    try:
        cleaned = text.replace('\n', ' ').replace('\t', ' ')
        cleaned = re.sub(r',\s*}', '}', cleaned)   # trailing comma
        cleaned = re.sub(r',\s*]', ']', cleaned)   # trailing comma in array
        start = cleaned.find('{'); end = cleaned.rfind('}')+1
        if start >= 0 and end > start:
            return json.loads(cleaned[start:end])
    except: pass
    # 3차: 핵심 필드만 정규식 추출
    try:
        lv = re.search(r'"severity_level"\s*:\s*"(level_[1-6]|null)"', text)
        crim = re.search(r'"criminal"\s*:\s*(true|false)', text, re.I)
        if lv:
            return {
                'severity_level': lv.group(1) if lv.group(1) != 'null' else None,
                'criminal': crim.group(1).lower() == 'true' if crim else False,
                'cited_articles': [],
                'reasoning': 'regex fallback parsing',
                'penalties': []
            }
    except: pass
    return None

# ── API Fallback ──
from agent_extras import LawAPIFallback, r_agent_with_fallback
api_fb = LawAPIFallback()

# ── 데이터 ──
CORE_ARTICLES = load_json(JSON_DIR / 'core_articles.json')
# ★ core_articles + v3 KG 확장 노드 ID 모두 포함 (M1 Citation Accuracy 공정성)
VALID_ARTICLE_IDS = {a['id'] for a in CORE_ARTICLES}
# v3 확장 노드: 시행령 별표, 교통안전법 항별, 교통사고특례법, 심각도 가이드 등
_v3_ids = {'RTAE_T7','RTAE_T8','RTAE_T10','RTAR_T28',
           'TSA_54','TSA_55_3','TSA_55_4',
           'TASA_3','TASA_4','SPCA_5_3','SPCA_5_11',
           'SGT_SEVERITY_GUIDE',
           'FINE_L1_FREIGHT','FINE_L2_FREIGHT','FINE_L3_FREIGHT','FINE_L4_FREIGHT'}
VALID_ARTICLE_IDS |= _v3_ids
SCENARIOS = load_json(SCENARIO_DIR / 'scenarios_100.json')

print(f'✅ Neo4j: {run_q("MATCH (n) RETURN count(n) AS c")[0]["c"]} nodes')
print(f'✅ 시나리오: {len(SCENARIOS)}개 | 조문: {len(CORE_ARTICLES)}개')

✅ Neo4j: 65 nodes
✅ 시나리오: 100개 | 조문: 15개


## Flat RAG 스토어 — 법령 원문만 사용
S2의 공정성을 위해 법령 조문 원문만 RAG 코퍼스로 사용.
SGT, 위반매핑, 벌칙 정답 등은 제외 (정답 포함 시 S2에 불공정 편향).

In [2]:
# ══════════════════════════════════════════
# S2 Flat RAG — 03에서 구축한 FAISS 인덱스 로드
# ══════════════════════════════════════════
import faiss
import numpy as np
from openai import OpenAI

oai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
EMBED_MODEL = 'text-embedding-3-small'

# 03에서 저장한 인덱스·청크·메타 로드
faiss_index = faiss.read_index(str(KG_DIR / 'faiss_index.bin'))
rag_chunks = load_json(KG_DIR / 'rag_chunks.json')
rag_meta = load_json(KG_DIR / 'rag_metadata.json')
print(f'✅ FAISS 로드: {faiss_index.ntotal}건, dim={faiss_index.d}')
print(f'✅ RAG 청크: {len(rag_chunks)}건, 메타: {len(rag_meta)}건')

def rag_search(query, top_k=5):
    """FAISS 기반 시맨틱 검색 (text-embedding-3-small)."""
    resp = oai_client.embeddings.create(input=[query], model=EMBED_MODEL)
    q_emb = np.array([resp.data[0].embedding], dtype='float32')
    faiss.normalize_L2(q_emb)
    scores, indices = faiss_index.search(q_emb, top_k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx < 0: continue
        results.append({
            'chunk': rag_chunks[idx],
            'meta': rag_meta[idx],
            'score': float(score)
        })
    return results

# 로드 확인
test = rag_search('과속 벌칙', top_k=2)
print(f'검색 테스트: {len(test)}건 반환 ✅')


✅ FAISS 로드: 46건, dim=1536
✅ RAG 청크: 46건, 메타: 46건
검색 테스트: 2건 반환 ✅


## IRAC-V 5 에이전트 (★ V-Agent 수정)

In [3]:
def i_agent(scenario):
    """I-Agent: 법적 쟁점 추출 — description에서 사실관계 파악.
    초과속도 계산, 위반유형 식별, 가중사유 인식.
    """
    prompt = f"""한국 도로교통법 전문가로서 아래 DTG 이벤트의 법적 쟁점을 추출하세요.

[이벤트] {scenario['description']}
위반 유형: {', '.join(scenario['violation_types'])}
{'어린이보호구역 내 발생' if scenario.get('school_zone') else ''}

[지침]
- 과속인 경우, 제한속도와 실제속도에서 초과속도(km/h)를 계산하세요.
- 화물차(4톤 초과) 법정 제한속도: 고속도로 80km/h, 자동차전용도로 80km/h.
- 제한속도가 명시되지 않은 경우, 도로유형에서 추론하세요.

JSON만 응답:
{{"issues":[{{"issue_type":"speeding","facts":"설명","speed_over_km":숫자,"aggravating":["school_zone" 등]}}]}}"""
    result = parse_json(call_llm(prompt, max_tokens=500))
    if result and 'issues' in result: return result
    return {'issues':[{'issue_type':scenario['violation_types'][0],
            'facts':scenario['description'],
            'speed_over_km': None,
            'aggravating':['school_zone'] if scenario.get('school_zone') else []}]}

def r_agent(issues):
    """R-Agent: KG에서 관련 법규 검색.
    I-Agent가 추출한 쟁점(위반유형, 초과속도)으로 KG 쿼리.
    ★ speed_over는 I-Agent가 추론한 값 사용 (시나리오 정답 참조 없음).
    """
    return r_agent_with_fallback(issues, run_q, api_fallback=api_fb, min_articles=2)

def a_agent(issues, rules):
    """A-Agent: 법리 포섭 분석 — 쟁점+법규로 심각도 추론."""
    ctx = []
    for rule in rules.get('rules', []):
        for art in rule['articles']:
            ctx.append(f"[{art.get('id','')}] {art.get('jo',art.get('jo_num',''))} "
                       f"{art.get('title','')}: {art.get('content','')[:200]}")
        # ★ 심각도 분류 기준표 (직접 조회가 아닌 기준 텍스트 제공)
        if rule.get('severity_guide'):
            for art in rule['severity_guide'].get('articles', []):
                ctx.append(f"[분류기준:{art.get('id','')}] {art.get('title','')}: {art.get('content','')}")

        for rel in rule.get('related_articles', []):
            ctx.append(f"[참조] {rel['id']} {rel.get('jo','')}: {rel.get('content','')[:150]}")

    prompt = f"""한국 도로교통법 전문가로서 아래 쟁점을 법규에 포섭 분석하세요.

[쟁점] {json.dumps(issues, ensure_ascii=False)}

[관련 법규 — KG 검색 결과]
{chr(10).join(ctx)}

[판정 지침]
- 초과속도가 계산되었으면 해당 구간의 벌칙을 적용하세요.
- 80km/h 초과: 형사처벌(제154조), 100km/h 초과: 가중처벌(제153조)
- 과속이 아닌 위반은 severity_level을 null로 판정하세요.
- 심각도: level_1(가장 경미) ~ level_6(가장 심각)

JSON만 응답:
{{"application":{{"severity_level":"level_1~6 또는 null","severity_label":"경미~극히위험 또는 null",
  "criminal":true/false,"legal_basis":"조문","cited_articles":["RTA_17"],
  "reasoning":"초과속도 계산 및 포섭 논증","penalties":["벌칙"]}}}}"""
    result = parse_json(call_llm(prompt, max_tokens=800))
    if result and 'application' in result: return result
    return {'application':{'severity_level':'unknown','reasoning':'파싱실패',
                           'cited_articles':[],'criminal':False}}


## ★ FIX 2: V-Agent — 정답지(GT) 제거, 내부 일관성 검증만
기존: scenario['expected']와 비교 → 데이터 누수 (Stage 4만 불이익)
수정: SGT 테이블 기준 내부 일관성만 검증

In [4]:
def v_agent(issues, rules, application, scenario):
    """V-Agent: 순수 내부 일관성 검증.
    
    ★ 정답(scenario.expected, scenario.speed_over) 일체 참조 안 함.
    I-Agent 추출 사실 ↔ A-Agent 판정 간 일관성만 검증.
    """
    app = application.get('application', {})
    errors = []
    
    # 1) Citation 검증: 인용 조문이 KG에 실존하는지
    cited = app.get('cited_articles', [])
    invalid_citations = [a for a in cited if a not in VALID_ARTICLE_IDS and not a.startswith('API_')]
    if invalid_citations:
        errors.append(f'존재하지 않는 조문: {invalid_citations}')
    
    # 2) I-Agent 추출 초과속도 ↔ A-Agent 판정 일관성
    predicted_lv = app.get('severity_level', '')
    speed_over = None
    for issue in issues.get('issues', []):
        so = issue.get('speed_over_km')
        if so and so > 0:
            speed_over = so
            break
    
    # ★ 내부 일관성 검증 (정답 직접 참조 없음)
    # A-Agent가 판정 실패(unknown)인데 과속 쟁점이 있으면 오류
    if predicted_lv in ('unknown','undetermined','',None) and speed_over and speed_over > 0:
        errors.append(f'판정 실패: 초과속도 {speed_over}km/h인데 심각도 미판정')
    
    # 형사/행정 일관성: level_5,6이면 criminal=True여야 함
    if predicted_lv in ('level_5','level_6') and not app.get('criminal', False):
        errors.append(f'형사 불일치: {predicted_lv}은 형사처벌이어야 함')
    elif predicted_lv in ('level_1','level_2','level_3','level_4') and app.get('criminal', False):
        errors.append(f'행정 불일치: {predicted_lv}은 행정처분이어야 함')
    
    # 3) 형사/행정 일관성
    if predicted_lv in SEVERITY_TABLE:
        predicted_criminal = app.get('criminal', False)
        expected_criminal = SEVERITY_TABLE[predicted_lv]['criminal']
        if predicted_criminal != expected_criminal:
            errors.append(f'형사/행정 불일치: {predicted_lv}은 {"형사" if expected_criminal else "행정"}')
    
    valid_cited = sum(1 for a in cited if a in VALID_ARTICLE_IDS or a.startswith('API_'))
    
    return {
        'valid': len(errors) == 0,
        'errors': errors,
        'citation_accuracy': valid_cited / max(len(cited), 1),
        
    }


def c_agent(issues, application, verification):
    """C-Agent: 최종 결론 통합.
    
    I-Agent(쟁점) + A-Agent(포섭) + V-Agent(검증) 결과를 종합하여
    최종 법적 판정을 구성합니다.
    """
    app = application.get('application', {})
    conclusion = {
        'severity_level': app.get('severity_level'),
        'severity_label': app.get('severity_label'),
        'criminal': app.get('criminal', False),
        'legal_basis': app.get('legal_basis', ''),
        'cited_articles': app.get('cited_articles', []),
        'reasoning': app.get('reasoning', ''),
        'penalties': app.get('penalties', []),
        'verification_passed': verification.get('valid', False),
        'verification_errors': verification.get('errors', []),
        'citation_accuracy': verification.get('citation_accuracy', 0),
    }
    
    # I-Agent에서 추출한 초과속도 정보 추가
    for issue in issues.get('issues', []):
        if issue.get('speed_over_km') is not None:
            conclusion['speed_over_km'] = issue['speed_over_km']
            break
    
    # V-Agent 검증 실패 시 결론에 경고 추가
    if not verification.get('valid', False):
        conclusion['warning'] = f"V-Agent 검증 미통과: {verification.get('errors', [])}"
    
    return {'conclusion': conclusion}

print('✅ C-Agent (결론 통합) 정의 완료')


✅ C-Agent (결론 통합) 정의 완료


## Stage 1: Vanilla LLM

In [5]:
# Stage 프롬프트는 각 함수 내에 정의 (S1→S2→S3→S4 정보량 증가)
# ★ v4: S1 프롬프트 강화 — 공정한 비교를 위해 동일 수준의 지시 제공
#   차이점은 오직 "외부 컨텍스트 제공 여부"만

def stage1_vanilla(scenario):
    """Stage 1: Vanilla LLM — 외부 컨텍스트 없이 LLM 내부 지식만 사용.
    
    ★ v4 수정: 프롬프트를 S2/S3과 동일 수준으로 강화.
    - 참조해야 할 법령명, 판정 기준 구조, 출력 형식을 동일하게 안내
    - 단, 실제 법령 텍스트나 KG 데이터는 제공하지 않음
    - 이를 통해 '프롬프트 품질 차이'가 아닌 '외부 지식 제공 효과'를 측정
    """
    start = time.time()
    prompt = f"""당신은 한국 도로교통법 전문가입니다.
아래 화물차 DTG 위험운전 이벤트에 대해 법적 심각도를 판정하세요.

[DTG 이벤트]
{scenario['description']}

[판정 지침]
1. 한국 도로교통법(제17조 속도제한, 제49조 준수사항 등), 동법 시행령(별표 7 벌점, 별표 8 범칙금),
   제156조(벌칙), 제154조(벌칙), 제153조(벌칙) 등을 참조하여 판정하세요.
2. 화물차(4톤 초과)의 법정 제한속도를 고려하세요:
   - 고속도로: 80km/h (승용차 100~110km/h와 다름)
   - 자동차전용도로: 80km/h
   - 일반도로: 도로별 지정 제한속도
3. 초과속도를 산출한 후, 해당 초과속도 구간에 맞는 심각도(level_1~level_6)를 판정하세요.
   - level_1: 가장 경미 (범칙금만)
   - level_2~3: 범칙금 + 벌점
   - level_4: 위험 (높은 범칙금 + 높은 벌점)
   - level_5~6: 형사처벌 대상
4. 과속이 아닌 위반(급가속, 급감속 등)은 severity_level을 null로 판정하세요.
5. 반드시 적용 법조문을 인용하세요.

JSON만 응답:
{{"severity_level":"level_1~6 또는 null","criminal":true/false,"legal_basis":"적용 조문",
"cited_articles":["조문ID"],"reasoning":"판단 근거 3문장","penalties":["벌칙 내용"]}}"""
    result = parse_json(call_llm(prompt, max_tokens=800))
    elapsed = time.time() - start
    if not result:
        result = {'severity_level':'unknown','criminal':False,'cited_articles':[],'reasoning':'파싱실패'}
    return {'scenario_id':scenario['id'],'stage':'stage1_vanilla',
            'conclusion':result,'elapsed_sec':round(elapsed,2)}


## Stage 2: Flat RAG

In [6]:
def stage2_flat_rag(scenario):
    """Stage 2: Flat RAG — S1과 동일 지시 + 검색된 법령 원문 텍스트 제공.
    
    S1과의 차이: 법령 원문 chunk가 컨텍스트로 추가됨
    S3과의 차이: 심각도 구간 매핑, 벌칙 정보 등 구조화 데이터 없음
    """
    start = time.time()
    query = scenario['description'] + ' ' + ' '.join(scenario.get('violation_types',[]))
    if scenario.get('school_zone'): query += ' 어린이보호구역'
    retrieved = rag_search(query, top_k=7)
    ctx = '\n\n'.join(f"[{r['meta']['id']}] {r['meta'].get('jo_num','')} "
                      f"{r['meta'].get('title','')}: {r['chunk'][:300]}" for r in retrieved)

    prompt = f"""당신은 한국 도로교통법 전문가입니다.
아래 화물차 DTG 위험운전 이벤트에 대해 법적 심각도를 판정하세요.
반드시 아래 제공된 참고 법령을 근거로 판단하세요.

[DTG 이벤트]
{scenario['description']}

[참고 법령 (검색 결과)]
{ctx}

[판정 지침]
1. 위 참고 법령과 당신의 법률 지식을 활용하여 판정하세요.
2. 화물차(4톤 초과) 법정 제한속도: 고속도로 80km/h, 자동차전용도로 80km/h.
3. 초과속도를 산출한 후, 심각도(level_1~level_6)를 판정하세요.
   - level_1: 가장 경미 ~ level_6: 가장 심각 (level_5~6은 형사처벌)
4. 과속이 아닌 위반(급가속, 급감속 등)은 severity_level을 null로 판정하세요.
5. 반드시 적용 법조문을 인용하세요.

JSON만: {{"severity_level":"level_1~6 또는 null","criminal":true/false,"legal_basis":"적용 조문",
"cited_articles":["조문ID"],"reasoning":"판단 근거 3문장","penalties":["벌칙 내용"]}}"""
    result = parse_json(call_llm(prompt, max_tokens=800))
    elapsed = time.time() - start
    if not result:
        result = {'severity_level':'unknown','criminal':False,'cited_articles':[],'reasoning':'파싱실패'}
    return {'scenario_id':scenario['id'],'stage':'stage2_flat_rag','conclusion':result,
            'retrieved_chunks':len(retrieved),'elapsed_sec':round(elapsed,2)}


## Stage 3: Single-Agent GraphRAG

In [7]:
def stage3_graphrag(scenario):
    """Stage 3: Single-Agent GraphRAG — KG 법조문+관계+벌칙기준 제공.
    
    ★ 핵심: SeverityLevel 직접 조회 없음.
    KG는 법조문과 조문 간 관계, 벌칙 기준 텍스트를 제공하고,
    LLM이 직접 초과속도를 계산하고 벌칙 기준에 대입하여 심각도를 추론.
    
    S2와의 차이: KG가 조문 간 관계(위반→정의→벌칙)를 구조적으로 제공
    S4와의 차이: S3은 모든 정보를 한 프롬프트에 넣고 1회 처리
    """
    start = time.time()
    ctx = []

    # 1) 위반유형 → 관련 법조문 (KG 관계 추적)
    for vtype in scenario.get('violation_types',[]):
        for a in run_q("MATCH (h:HazardousBehavior {name:$v})-[:VIOLATES]->(a:LegalArticle) "
                       "RETURN a.id AS id, a.jo_num AS jo, a.title AS title, a.content AS content "
                       "ORDER BY a.id", {'v':vtype}):
            ctx.append(f"[{a['id']}] {a.get('jo','')} {a.get('title','')}: {a.get('content','')[:200]}")

        # Multi-hop: 위반→정의조문→벌칙조문
        for r in run_q("MATCH (h:HazardousBehavior {name:$v})-[:VIOLATES]->(a)-[:RELATED_TO]->(a2) "
                       "RETURN DISTINCT a2.id AS id, a2.jo_num AS jo, a2.title AS title, a2.content AS content",
                       {'v':vtype}):
            ctx.append(f"[참조:{r['id']}] {r.get('jo','')} {r.get('title','')}: {r.get('content','')[:200]}")

    # 2) 벌칙 기준표 (별표) — 범위만 제공, 정답 조회 아님
    penalty_tables = run_q("MATCH (a:LegalArticle) WHERE a.type='penalty_table' "
                           "RETURN a.id AS id, a.jo_num AS jo, a.title AS title, a.content AS content")
    for p in penalty_tables:
        ctx.append(f"[벌칙기준:{p['id']}] {p.get('jo','')} {p.get('title','')}: {p.get('content','')}")

    # 3) 가중조건 (어린이보호구역 등)
    if scenario.get('school_zone'):
        aggr = run_q("MATCH (af:AggravatingFactor) RETURN af.name AS name, af.description AS desc")
        for a in aggr:
            ctx.append(f"[가중조건] {a.get('name','')}: {a.get('desc','')}")

    # ★ SeverityLevel 직접 조회 없음! LLM이 직접 추론해야 함

    prompt = f"""당신은 한국 도로교통법 전문가입니다.
아래 화물차 DTG 위험운전 이벤트에 대해 Knowledge Graph에서 검색된 법령 정보를 참조하여
법적 심각도를 판정하세요.

[DTG 이벤트]
{scenario['description']}

[KG 검색 결과 — 관련 법조문 및 벌칙 기준]
{chr(10).join(ctx)}

[판정 지침]
1. 이벤트에서 초과속도를 직접 계산하세요 (실제속도 - 제한속도).
   화물차(4톤 초과) 법정 제한속도: 고속도로 80km/h, 자동차전용도로 80km/h.
2. 계산된 초과속도를 위 벌칙 기준에 대입하여 심각도를 판정하세요.
   - KG 검색 결과의 '심각도 분류 기준'을 참조하여 초과속도 구간에 맞는 level을 판정하세요.
   - 기준이 없으면: level_1(가장 경미, 범칙금만) ~ level_6(가장 심각, 형사처벌)
3. 과속이 아닌 위반은 severity_level을 null로 판정하세요.
4. 반드시 적용 법조문을 인용하세요.

JSON만: {{"severity_level":"level_1~6 또는 null","criminal":true/false,"legal_basis":"적용 조문",
"cited_articles":["조문ID"],"reasoning":"초과속도 계산 과정과 판단 근거","penalties":["벌칙 내용"]}}"""
    result = parse_json(call_llm(prompt, max_tokens=800))
    elapsed = time.time() - start
    if not result:
        result = {'severity_level':'unknown','criminal':False,'cited_articles':[],'reasoning':'파싱실패'}
    return {'scenario_id':scenario['id'],'stage':'stage3_graphrag','conclusion':result,
            'kg_context_items':len(ctx),'elapsed_sec':round(elapsed,2)}


## Stage 4: IRAC-V (V-Agent + Self-Healing)

In [8]:
def stage4_iracv(scenario, max_retries=2):
    """Stage 4: IRAC-V 멀티에이전트.
    
    ★ 정답 참조 없음. 모든 추론은 description에서 시작.
    I-Agent가 초과속도를 계산 → R-Agent가 KG에서 관련 법규 검색
    → A-Agent가 법리 포섭 → V-Agent가 내부 일관성 검증.
    
    S3과의 차이:
    - S3: 모든 KG 정보를 한 프롬프트에 → 1회 추론
    - S4: 5에이전트 역할 분담 → 단계별 추론 + 검증 + Self-Healing
    """
    start = time.time()
    issues = i_agent(scenario)
    rules = r_agent(issues)        # ★ I-Agent 추론값으로 KG 쿼리
    application = a_agent(issues, rules)
    verification = v_agent(issues, rules, application, scenario)

    # V-Agent 통과 → 즉시 확정
    best_application = json.loads(json.dumps(application))
    retries = 0

    if not verification.get('valid', False):
        kg_errors = [e for e in verification.get('errors',[])
                     if '판정 실패' in e or '형사 불일치' in e or '행정 불일치' in e]

        while kg_errors and retries < max_retries:
            retries += 1
            # 재시도: I-Agent 결과는 유지, R-Agent부터 다시
            rules_retry = r_agent(issues)
            application = a_agent(issues, rules_retry)
            verification = v_agent(issues, rules_retry, application, scenario)

            if verification.get('valid', False):
                best_application = json.loads(json.dumps(application))
                break

            retry_lv = application.get('application',{}).get('severity_level','')
            if retry_lv and retry_lv not in ('unknown','undetermined',''):
                best_application = json.loads(json.dumps(application))

            kg_errors = [e for e in verification.get('errors',[])
                         if '판정 실패' in e or '형사 불일치' in e or '행정 불일치' in e]

    conclusion = c_agent(issues, best_application, verification)
    elapsed = time.time() - start
    api_used = any(r.get('api_fallback_used',False) for r in rules.get('rules',[]))

    return {'scenario_id':scenario['id'],'stage':'stage4_iracv','issues':issues,
            'rules_summary':{r['issue_type']:[a.get('id','') for a in r['articles']]
                             for r in rules.get('rules',[])},
            'conclusion':conclusion.get('conclusion',{}),
            'verification':verification,'retries':retries,
            'api_fallback_used':api_used,'elapsed_sec':round(elapsed,2)}

print('✅ Stage 4 (IRAC-V) 정의 완료')


✅ Stage 4 (IRAC-V) 정의 완료


## 전체 실험 실행 (4 stages × N회 반복)
3회 반복으로 통계적 안정성 확보
⚠️ N_REPEAT=1로 먼저 테스트 후 3으로 확대

In [9]:
def _normalize_severity(val):
    if val is None: return None
    s = str(val).strip().lower()
    if s in ('none','null','unknown',''): return None
    return str(val)

# ★ v3: 확대된 시나리오 사용
N = min(len(SCENARIOS), 100)
N_REPEAT = 3   # ★ 반복 횟수 (비용 절감 시 1)
test_scenarios = SCENARIOS[:N]
print(f'✅ 실험 시나리오: {N}건 × {N_REPEAT}회 반복')

stage_fns = [
    ('stage1', stage1_vanilla),
    ('stage2', stage2_flat_rag),
    ('stage3', stage3_graphrag),
    ('stage4', stage4_iracv),
]

all_runs = []
for run_id in range(N_REPEAT):
    print(f'\n{"="*60}')
    print(f'  반복 {run_id+1}/{N_REPEAT}')
    print(f'{"="*60}')
    run_results = {s: [] for s, _ in stage_fns}
    for i, sc in enumerate(test_scenarios):
        print(f'\n  [{i+1}/{N}] {sc["id"]}: {sc["description"][:40]}...')
        for skey, fn in stage_fns:
            r = fn(sc)
            run_results[skey].append(r)
            pred = r.get('conclusion',{}).get('severity_level','?')
            expected = sc['expected'].get('severity_level')
            match = '✅' if _normalize_severity(pred) == _normalize_severity(expected) else '❌'
            print(f'    {skey}: {pred} {match}', end='')
        print()
        time.sleep(0.5)
    all_runs.append(run_results)
    for skey in run_results:
        (RESULTS_DIR / skey).mkdir(parents=True, exist_ok=True)
        save_json(run_results[skey], RESULTS_DIR / skey / f'results_{N}scenarios_run{run_id+1}.json')

print(f'\n=== 전체 완료: {N}건 × {len(stage_fns)} stages × {N_REPEAT}회 ===')


✅ 실험 시나리오: 100건 × 3회 반복

  반복 1/3

  [1/100] S001: 화물차(4톤 초과)가 제한속도 60km/h 일반도로에서 75km/h로 주...
    stage1: level_2 ❌    stage2: level_2 ❌    stage3: level_1 ✅    stage4: level_1 ✅

  [2/100] S002: 화물차(4톤 초과)가 제한속도 60km/h 일반도로에서 95km/h로 주...
    stage1: level_5 ❌    stage2: level_5 ❌    stage3: level_5 ❌    stage4: level_2 ✅

  [3/100] S003: 화물차(4톤 초과)가 제한속도 60km/h 일반도로에서 110km/h로 ...
    stage1: level_6 ❌    stage2: level_6 ❌    stage3: level_6 ❌    stage4: level_5 ❌

  [4/100] S004: 화물차(4톤 초과)가 제한속도 60km/h 일반도로에서 130km/h로 ...
    stage1: level_6 ❌    stage2: level_6 ❌    stage3: level_6 ❌    stage4: level_5 ❌

  [5/100] S005: 화물차(4톤 초과)가 제한속도 60km/h 일반도로에서 150km/h로 ...
    stage1: level_6 ❌    stage2: level_6 ❌    stage3: level_6 ❌    stage4: level_5 ✅

  [6/100] S006: 화물차(4톤 초과)가 제한속도 60km/h 일반도로에서 170km/h로 ...
    stage1: level_6 ✅    stage2: level_6 ✅    stage3: level_6 ✅    stage4: level_6 ✅

  [7/100] S007: 화물차(4톤 초과)가 제한속도 50km/h 일반도로에서 65km/h로 주...
    stage1: level_2 ❌    stag

## 평가 (평균±표준편차)

In [12]:
from sklearn.metrics import f1_score

def _extract_amount(text):
    text = str(text).replace(',','').replace(' ','')
    m = re.search(r'(\d+)만원', text)
    if m: return int(m.group(1)) * 10000
    m = re.search(r'(\d+)원', text)
    if m: return int(m.group(1))
    m = re.search(r'(\d+)', text)
    if m and int(m.group(1)) > 100: return int(m.group(1))
    return None

def _calc_multihop_recall(results_list, scenarios_list):
    """★ v3: Multi-hop Recall — KG 관계 추론 효용 측정.
    
    각 위반유형의 필수 인용 조문(정의조문+벌칙조문)을 계산하고,
    실제 인용된 조문의 Recall을 산출. KG 구조가 있으면
    Multi-hop 경로(위반→정의→벌칙)를 따라가므로 Recall이 높아짐.
    """
    scores = []
    for r, sc in zip(results_list, scenarios_list):
        cited = set(r.get('conclusion',{}).get('cited_articles',[]))
        cited_norm = {c for c in cited if not c.startswith('API_')}
        # API_ 접두사 조문도 정규화하여 포함
        for c in cited:
            if c.startswith('API_'):
                cited_norm.add(c)  # API 조문도 유효
        required = set()
        for vtype in sc.get('violation_types',[]):
            mapping = VIOLATION_MAPPING.get(vtype, {})
            required.update(mapping.get('definition_articles', []))
            if mapping.get('penalty_articles'):
                required.add(mapping['penalty_articles'][0])  # 최소 1개 벌칙조문
        if sc.get('school_zone'):
            required.add('RTA_12')
        if required:
            recall = len(cited_norm & required) / len(required)
            scores.append(recall)
    return round(np.mean(scores), 4) if scores else 0

def evaluate_run(results_list, scenarios_list):
    n = len(results_list)
    if n == 0: return {}
    y_true = [str(_normalize_severity(sc['expected'].get('severity_level'))) for sc in scenarios_list]
    y_pred = [str(_normalize_severity(r.get('conclusion',{}).get('severity_level'))) for r in results_list]
    sev_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    sev_acc = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp) / n
    crim_acc = sum(1 for r, sc in zip(results_list, scenarios_list)
                   if bool(r.get('conclusion',{}).get('criminal')) == bool(sc['expected'].get('criminal'))) / n
    cite_accs = []
    for r in results_list:
        cited = r.get('conclusion',{}).get('cited_articles',[])
        if not cited: cite_accs.append(1.0)
        else: cite_accs.append(sum(1 for a in cited if a in VALID_ARTICLE_IDS or a.startswith('API_'))/len(cited))
    m1 = np.mean(cite_accs)
    pen_ok, pen_tot = 0, 0
    for r, sc in zip(results_list, scenarios_list):
        exp_fine = sc['expected'].get('fine_amount')
        if exp_fine and exp_fine > 0:
            pen_tot += 1
            for pp in r.get('conclusion',{}).get('penalties',[]):
                ext = _extract_amount(pp)
                if ext and ext == exp_fine: pen_ok += 1; break
            else:
                if f'{exp_fine//10000}만' in ' '.join(str(p) for p in r.get('conclusion',{}).get('penalties',[])): pen_ok += 1
    m2 = pen_ok / max(pen_tot, 1)
    # ★ v3: Multi-hop Recall 추가
    mh_recall = _calc_multihop_recall(results_list, scenarios_list)
    return {'sev_f1':round(sev_f1,4),'sev_acc':round(sev_acc,4),'crim_acc':round(crim_acc,4),
            'cite_acc':round(m1,4),'halluc':round(1-m1,4),'pen_acc':round(m2,4),
            'mh_recall':mh_recall,
            'avg_time':round(np.mean([r.get('elapsed_sec',0) for r in results_list]),2)}

stage_keys = [s for s, _ in stage_fns]
metric_names = ['sev_f1','sev_acc','crim_acc','cite_acc','halluc','pen_acc','mh_recall','avg_time']
metric_labels = {'sev_f1':'Severity F1','sev_acc':'Severity Acc','crim_acc':'Criminal Acc',
                 'cite_acc':'Citation Acc','halluc':'Hallucination','pen_acc':'Penalty Acc',
                 'mh_recall':'Multi-hop Recall','avg_time':'Avg Time'}

stage_metrics = {s: {m: [] for m in metric_names} for s in stage_keys}
for run_id, run_results in enumerate(all_runs):
    for skey in stage_keys:
        ev = evaluate_run(run_results[skey], test_scenarios)
        for m in metric_names: stage_metrics[skey][m].append(ev.get(m,0))

print(f'\n=== {N_REPEAT}회 반복 결과 (평균±표준편차) ===\n')
header = f'{"메트릭":<18s}'
for s in stage_keys: header += f' | {s:>14s}'
print(header); print('-'*len(header))
for m in metric_names:
    row = f'{metric_labels[m]:<18s}'
    for s in stage_keys:
        vals = stage_metrics[s][m]
        mean = np.mean(vals); std = np.std(vals)
        if 'time' in m: row += f' | {mean:>9.1f}s±{std:.1f}'
        elif N_REPEAT > 1: row += f' | {mean:>8.1%}±{std:.1%}'
        else: row += f' |       {mean:>8.1%}'
    print(row)

print(f'\n=== 가설 검증 ===')
for m in ['sev_f1','sev_acc','mh_recall']:
    v = [np.mean(stage_metrics[s][m]) for s in ['stage1','stage2','stage3','stage4']]
    s4_best = v[3] == max(v)
    print(f'  {metric_labels[m]}: {[f"{x:.1%}" for x in v]} → S4최고:{"✅" if s4_best else "❌"}')

# ★ v3: KG 효용 검증
s2_mh = np.mean(stage_metrics['stage2']['mh_recall'])
s3_mh = np.mean(stage_metrics['stage3']['mh_recall'])
print(f'\n=== KG 구조 효용 (S2→S3) ===')
print(f'  Multi-hop Recall: S2={s2_mh:.1%} → S3={s3_mh:.1%} (Δ={s3_mh-s2_mh:+.1%}p)')
print(f'  Severity F1: S2={np.mean(stage_metrics["stage2"]["sev_f1"]):.1%} → '
      f'S3={np.mean(stage_metrics["stage3"]["sev_f1"]):.1%}')



=== 3회 반복 결과 (평균±표준편차) ===

메트릭                |         stage1 |         stage2 |         stage3 |         stage4
--------------------------------------------------------------------------------------
Severity F1        |    22.4%±0.5% |    24.2%±0.0% |    27.1%±0.0% |    63.3%±1.4%
Severity Acc       |    31.3%±0.5% |    33.0%±0.0% |    34.0%±0.0% |    64.3%±1.2%
Criminal Acc       |    54.7%±0.5% |    55.0%±0.0% |    50.7%±0.5% |    75.7%±1.2%
Citation Acc       |     1.3%±0.5% |   100.0%±0.0% |   100.0%±0.0% |    96.3%±0.4%
Hallucination      |    98.7%±0.5% |     0.0%±0.0% |     0.0%±0.0% |     3.7%±0.4%
Penalty Acc        |    11.5%±2.7% |    48.1%±2.8% |    13.7%±2.0% |    56.3%±2.0%
Multi-hop Recall   |     0.0%±0.0% |    30.9%±0.2% |    53.0%±0.2% |    70.7%±0.6%
Avg Time           |       6.4s±0.1 |       6.7s±0.0 |       7.4s±0.0 |      11.7s±0.2

=== 가설 검증 ===
  Severity F1: ['22.4%', '24.2%', '27.1%', '63.3%'] → S4최고:✅
  Severity Acc: ['31.3%', '33.0%', '34.0%', '64.3%'] 

## 결과 저장

In [13]:
# 대표 결과 저장 (마지막 반복 또는 첫 반복)
rep_idx = 0  # 대표 반복 인덱스
for skey in stage_keys:
    save_json(all_runs[rep_idx][skey], RESULTS_DIR / skey / f'results_{N}scenarios.json')

# 통합 평가 저장
eval_summary = {
    'evals': {s: {metric_labels[m]: {'mean':float(np.mean(stage_metrics[s][m])),
                                     'std':float(np.std(stage_metrics[s][m]))}
                  for m in metric_names} for s in stage_keys},
    'n_scenarios': N, 'n_repeats': N_REPEAT,
    'timestamp': datetime.datetime.now().isoformat(), 'model': LLM_MODEL,
}
save_json(eval_summary, RESULTS_DIR / 'evaluation' / 'eval_summary_repeated.json')

# 05와 호환되는 형식도 저장
compat_evals = {}
for skey in stage_keys:
    compat_evals[skey] = {
        'stage': skey, 'n': N,
        'M1_citation_accuracy': float(np.mean(stage_metrics[skey]['cite_acc'])),
        'M2_penalty_accuracy': float(np.mean(stage_metrics[skey]['pen_acc'])),
        'M3_severity_f1_macro': float(np.mean(stage_metrics[skey]['sev_f1'])),
        'M4_reasoning_quality': None,
        'M5_hallucination_rate': float(np.mean(stage_metrics[skey]['halluc'])),
        'M6_multihop_recall': float(np.mean(stage_metrics[skey]['mh_recall'])),
        'severity_accuracy': float(np.mean(stage_metrics[skey]['sev_acc'])),
        'criminal_accuracy': float(np.mean(stage_metrics[skey]['crim_acc'])),
        'avg_time_sec': float(np.mean(stage_metrics[skey]['avg_time'])),
    }
save_json({'evals':compat_evals,'n_scenarios':N,'n_repeats':N_REPEAT,
           'timestamp':datetime.datetime.now().isoformat(),'model':LLM_MODEL},
          RESULTS_DIR / 'evaluation' / 'eval_summary.json')
print(f'\n→ 다음: 05_analysis_and_paper.ipynb')


✅ 저장: results\stage1\results_100scenarios.json
✅ 저장: results\stage2\results_100scenarios.json
✅ 저장: results\stage3\results_100scenarios.json
✅ 저장: results\stage4\results_100scenarios.json
✅ 저장: results\evaluation\eval_summary_repeated.json
✅ 저장: results\evaluation\eval_summary.json

→ 다음: 05_analysis_and_paper.ipynb
